# 16 — Bidding on Part 2's forecasts (Part 3, Stage 6)

**Does:** firms choose tomorrow's bid on what they know today (Part 2's forecasts); the market then clears on what actually happens. How much of the perfect-information profit do they capture, and do forecast warning signs flag market-power days in advance?

**Input:** Part 2's `part3_forecast_package.csv` (demand median, 10% and 90% per state; forecast spare capacity and near-step flag) and `availability_forecast_test.csv` ("best day-ahead" solar, wind, hydro). **Runs:** `src/run_part3.py stage6` (6.3 min on 9 processes) → `strategic_forecast_daily.csv`.

**Plan:** `docs/part3_plan.md`, Stage 6. **Logged in:** `docs/model_log.md`, Step 27.

## Settings (Step 27)

| Setting | Value |
|---|---|
| Days | All 396 (1 Mar 2024 – 31 Mar 2025) |
| Firms | The two largest, **UP** and **Maharashtra**, each alone (everyone else bids cost); bid only, no withholding |
| Strategies compared | **perfect** (best bid for the actual day, upper bound) · **forecast** (best bid on the median forecast) · **robust** (best average profit over the 10%, median and 90% demand forecasts) · **persistence** (best bid for yesterday's actual day) · **cost** (adder 0) |
| Scoring | Each strategy's bid is fixed, then the market clears on the **actual** day: realised profit and consumer cost |
| Market-power day (6.5) | UP or Maharashtra, bidding with perfect foresight, raises the average price **≥ 5%** above competitive (decision A: comes from this run; the 24-firm Stage 5 run is deferred) |
| Warning signs (6.5) | Forecast spare capacity < 15%, forecast near-step flag (Part 2), either; each also computed from *actual* values to separate forecast error from a weak flag |

**Check before the run:** feeding the actual day in as every forecast gives the perfect-foresight bid and profit (`tests/test_bidding.py`).

**Hypotheses (written before the run):** forecast bids capture most (> 80%) of the perfect-information gain and beat persistence; robust bids earn a little less on average but lose less on bad days; forecast flags catch market-power days only partly (Stage 3: market power appears on slack days too).

In [1]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dispatch import PROCESSED
from forecast import load_forecast_package

crore = 1e7
path = PROCESSED / "strategic_forecast_daily.csv"
if not path.exists():
    raise FileNotFoundError("Run `.venv/bin/python src/run_part3.py stage6` first (~3 min)")
fc = pd.read_csv(path, parse_dates=["Date"])
pkg = load_forecast_package()
pkg_nat = pkg[pkg["Area"] == "National"].set_index("Date")

print(fc.groupby(["Firm", "Strategy"]).size().unstack())
print("Unserved MW, max over all runs:", fc["Unserved_MW"].max())

Strategy        cost  forecast  perfect  persistence  robust
Firm                                                        
NR UP            396       396      396          396     396
WR Maharashtra   396       396      396          396     396
Unserved MW, max over all runs: 0.0


## 6.2–6.4 Realised profit by strategy

In [2]:
w = fc.pivot_table(index=["Date", "Firm"], columns="Strategy", values="Profit")
gain = w.sub(w["cost"], axis=0)                     # extra profit over bidding cost
a = fc.pivot_table(index=["Date", "Firm"], columns="Strategy", values="Adder")
rows = {}
for firm, g in gain.groupby(level="Firm"):
    perfect = g["perfect"].sum()
    rows[firm] = {f"{k} % of perfect gain": 100 * g[k].sum() / perfect for k in ["forecast", "robust", "persistence"]}
    rows[firm]["Perfect gain (Rs crore/h, avg)"] = g["perfect"].mean() / crore
    rows[firm]["Days perfect marks up"] = int((a.xs(firm, level="Firm")["perfect"] > 0).sum())
    for k in ["forecast", "robust", "persistence"]:
        rows[firm][f"Days {k} = perfect bid"] = int((a.xs(firm, level="Firm")[k] == a.xs(firm, level="Firm")["perfect"]).sum())
        rows[firm][f"Days {k} loses vs cost"] = int((g[k] < -1).sum())
summary = pd.DataFrame(rows).T
display(summary.T.round(1))

c = fc.pivot_table(index=["Date", "Firm"], columns="Strategy", values="Consumer_cost")
print("Consumer cost increase vs everyone at cost (%, average over days):")
display((100 * c.div(c["cost"], axis=0).sub(1)).groupby(level="Firm").mean().round(2))

worst = gain.groupby(level="Firm").quantile(0.05) / crore
print("Worst 5% of days, gain over cost (Rs crore/h):"); display(worst.round(3))

,NR UP,WR Maharashtra
forecast % of perfect gain,70.6,83.3
robust % of perfect gain,65.9,62.3
persistence % of perfect gain,42.5,62.4
"Perfect gain (Rs crore/h, avg)",0.5,0.7
Days perfect marks up,162.0,295.0
Days forecast = perfect bid,281.0,223.0
Days forecast loses vs cost,39.0,22.0
Days robust = perfect bid,153.0,93.0
Days robust loses vs cost,43.0,83.0
Days persistence = perfect bid,217.0,156.0


Consumer cost increase vs everyone at cost (%, average over days):


Strategy,cost,forecast,perfect,persistence,robust
Firm,,,,,
NR UP,0.0,17.55,16.91,12.85,19.11
WR Maharashtra,0.0,14.94,16.91,11.72,17.92


Worst 5% of days, gain over cost (Rs crore/h):


Strategy,cost,forecast,perfect,persistence,robust
Firm,,,,,
NR UP,0.0,-0.332,0.0,-0.537,-0.351
WR Maharashtra,0.0,-0.023,0.0,-0.086,-1.278


## 6.5 Do forecast warning signs flag market-power days?

In [3]:
p = fc.pivot_table(index="Date", columns=["Firm", "Strategy"], values="Price_avg")
uplift = pd.DataFrame({f: p[(f, "perfect")] / p[(f, "cost")] - 1 for f in fc["Firm"].unique()})
power = uplift.max(axis=1) > 0.05
nat = pkg_nat.reindex(power.index)

flags = {}
for when in ["forecast", "actual"]:
    tight = nat[f"Tightness_{when}"] < 0.15
    step = nat[f"Near_step_{when}"].astype(bool)
    flags[(when, "Spare < 15%")] = tight
    flags[(when, "Near step")] = step
    flags[(when, "Either")] = tight | step

def pr(flag, truth):
    tp = (flag & truth).sum()
    return {"Flagged days": int(flag.sum()), "Precision %": 100 * tp / max(flag.sum(), 1),
            "Recall %": 100 * tp / max(truth.sum(), 1)}
print(f"Market-power days (UP or Maharashtra raises the price >= 5% with perfect foresight): "
      f"{int(power.sum())} of {len(power)}")
display(pd.DataFrame({k: pr(v, power) for k, v in flags.items()}).T.round(1))

# Market-power days by forecast spare capacity
bands = pd.cut(nat["Tightness_forecast"], [0, 0.10, 0.15, 0.20, 0.30, 1.0])
display(pd.DataFrame({"Days": power.groupby(bands, observed=True).size(),
                      "Market-power %": 100 * power.groupby(bands, observed=True).mean()}).round(1))

Market-power days (UP or Maharashtra raises the price >= 5% with perfect foresight): 203 of 396


Flagged days  Precision %  Recall %
forecast Spare < 15%         131.0         46.6      30.0
         Near step            82.0         57.3      23.2
         Either              175.0         49.7      42.9
actual   Spare < 15%         119.0         46.2      27.1
         Near step            82.0         81.7      33.0
         Either              170.0         57.6      48.3

,Days,Market-power %
Tightness_forecast,,
"(0.0, 0.1]",52,69.2
"(0.1, 0.15]",89,33.7
"(0.15, 0.2]",67,53.7
"(0.2, 0.3]",111,73.9
"(0.3, 1.0]",77,24.7


## Stage 6 conclusions

**Checks:** no unserved power in any of the 3,960 runs; perfect-foresight profit never below profit at cost; cost-bid prices equal Part 2's `dispatch_weather_national_daily.csv` within Rs 0.84/MWh on all 396 days.

| Hypothesis | Result |
|---|---|
| Forecast bids capture > 80% of the perfect-information gain | ✅ Maharashtra 83%, ❌ UP 71% (same bid as perfect on 281 and 223 days) |
| Forecast bids beat persistence ("tomorrow = today") | ✅ by 20–28 points (persistence 43% and 62%) |
| Robust bids earn a little less but lose less on bad days | ❌ They earn less (66%, 62%) **and** lose more: Maharashtra's robust bid earns less than bidding cost on 83 days (forecast bid: 22), worst 5% of days −Rs 1.28 crore/h |
| Forecast flags catch market-power days only partly | ✅ "Either" flag: 43% of the 203 market-power days, half of flagged days right |

- **Part 2's forecasts are worth money to a strategic firm:** most of the perfect-foresight gain is captured, far more than with yesterday's best bid
- **Averaging over scenarios is not "robust":** profit is lopsided. In the 90% demand scenario a high bid pays enormously (near the cap), so the average favours high bids, and on an ordinary day the firm prices itself out. A cautious firm would need a downside-weighted rule (e.g. maximise the worst scenario), not the average
- **Forecast errors don't protect consumers:** consumer cost rises about as much on forecast bids as with perfect foresight (UP +17.6% vs +16.9%; Maharashtra +14.9% vs +16.9%). Robust bids raise it most (+18–19%)
- **Warning flags are weak because market power doesn't follow scarcity, not because of forecast error.** With actual values the flags catch only 48%. Market power is most common at 20–30% spare capacity (74% of days), less at 10–15% (34%) than at ≤ 10% (69%) or 15–20% (54%): the coal-cost steps, not tightness, decide it (Stage 3, Step 24). The near-step flag is precise when computed from actuals (82%) but forecast errors drop that to 57%
- **Caveats:** two firms, each alone (market-power days come from this run, not the deferred 24-firm Stage 5 equilibrium, so days where only a smaller firm has power are missed); bid only; the robust rule searches the candidate grid without the Rs 25 refinement